## LoRA로 한국어 질문·응답 학습하기

KoAlpaca의 한국어 질문·응답 사례를 이용해 Qwen 모델이 질문에 맞는 답변을 생성하도록 파인튜닝한다.
질문을 입력으로, 데이터에 담긴 응답을 학습 대상으로 삼아 답변의 내용과 표현 방식을 학습한다.
기본 모델의 가중치는 고정하고, 추가한 LoRA 어댑터의 가중치만 갱신한다.


### 패키지 설치

데이터 처리와 모델 학습에 필요한 패키지를 설치한다.


In [ ]:
%pip install pandas matplotlib scikit-learn
%pip install transformers peft accelerate datasets


설치한 패키지를 새로 불러오도록 `Kernel → Restart Kernel`로 커널을 재시작한다.
재시작 후에는 설치 셀을 건너뛰고, 「실행 환경 준비」의 코드 셀부터 순서대로 실행한다.


### 실행 환경 준비

CUDA GPU와 BF16(16비트 부동소수점 형식) 연산을 지원하는 환경에서 실행한다.
필요한 라이브러리를 불러오고 학습 결과의 저장 위치를 정한다.


In [ ]:
import gc
import json
from pathlib import Path

# 학습 결과를 저장할 기준 경로
OUTPUT_ROOT = Path("/workspace") if Path("/workspace").is_dir() else Path.cwd()

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    Trainer, TrainingArguments, set_seed,
)

SEED = 42
set_seed(SEED)


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다.")


### 파인튜닝할 모델

추가 학습의 출발점으로 약 15억 개 파라미터를 가진 `Qwen/Qwen2.5-1.5B-Instruct`를 사용한다.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

### 질문·응답 데이터 불러오기

모델이 따라 배울 한국어 질문·응답 데이터인 [KoAlpaca-v1.1a](https://huggingface.co/datasets/beomi/KoAlpaca-v1.1a)를 불러온다.
한 행에서 `instruction`은 질문, `output`은 학습할 응답이다.


In [ ]:
DATASET_ID = "beomi/KoAlpaca-v1.1a"
raw_dataset = load_dataset(
    DATASET_ID,

    # 공개 데이터 전체 구간.
    split="train",
)

# Dataset → DataFrame, 한 행은 질문·응답 한 쌍
raw_frame = raw_dataset.to_pandas()


### 학습·검증 분리

전체 데이터 중 일부를 선택해 학습용과 학습 전후 비교용 검증 데이터로 나눈다.


In [ ]:
examples = raw_frame[["instruction", "output"]].copy()

# 학습 전후 응답을 연결할 원본 행 번호
examples["source_row"] = examples.index

# 시드를 고정해 같은 행을 선택한다.
pool = examples.sample(n=576, random_state=SEED)

# 선택한 사례 중 64건을 검증용으로 나눈다.
train_frame, val_frame = train_test_split(
    pool, test_size=64, random_state=SEED
)

train_rows = train_frame.to_dict("records")
val_rows = val_frame.to_dict("records")
comparison_rows = val_rows[:5]
with pd.option_context("display.max_colwidth", None):
    display(train_frame[["source_row", "instruction", "output"]].head(3))


## 기본 모델의 응답 생성

### 모델과 토크나이저 불러오기

학습 전 답변을 확인할 기본 모델과, 문장을 모델의 입력으로 변환할 토크나이저를 준비한다.


In [ ]:
DTYPE = torch.bfloat16
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,

    # 모델 전체를 첫 번째 GPU(cuda:0)에 배치한다.
    device_map={"": 0},
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 모델과 생성 설정의 PAD를 토크나이저에 맞춘다.
base.config.pad_token_id = tokenizer.pad_token_id
base.generation_config.pad_token_id = tokenizer.pad_token_id


### 질문을 채팅 입력으로 만들기

모델과 함께 제공된 채팅 템플릿을 사용해, 지시문과 질문을 해당 모델의 입력 형식으로 변환한다.


In [ ]:
SYSTEM = "질문에 한국어로 답하세요. 핵심 내용을 간결하고 이해하기 쉽게 설명하세요."


def messages_for(question):
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]


def prompt_ids_for(question):
    return tokenizer.apply_chat_template(
        messages_for(question),

        # 정수 토큰 ID 목록을 반환한다.
        return_dict=False,

        # assistant의 응답 시작 표시를 붙인다.
        add_generation_prompt=True,
    )


### 응답 생성하기

검증 질문에 대한 학습 전후 모델의 답변을 비교하기 위한 생성 함수를 만든다.
모델의 기본 샘플링 설정을 사용하고, 응답 길이에 상한을 둔다. 샘플링에 따라 응답은 실행마다 달라질 수 있다.


In [ ]:
# 입력 뒤에 새로 생성할 응답의 최대 토큰 수
MAX_NEW_TOKENS = 256


# 응답 생성 중 기울기를 계산하지 않는다.
@torch.no_grad()
def generate_response(model, question):

    model.eval()
    ids = prompt_ids_for(question)
    if len(ids) + MAX_NEW_TOKENS > model.config.max_position_embeddings:
        raise ValueError("입력과 출력 예산이 모델의 문맥 길이를 넘습니다.")

    # 두 입력의 모양은 (1, 입력 토큰 수). attention_mask의 1은 사용할 위치다.
    inputs = {
        "input_ids": torch.tensor([ids], device=model.device),
        "attention_mask": torch.ones(
            (1, len(ids)), dtype=torch.long, device=model.device
        ),
    }
    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # 입력을 제외한 새 응답 토큰
    response_ids = generated[0, len(ids):].tolist()
    return {

        # skip_special_tokens=True: 역할·종료 등의 특수 토큰을 출력에서 제외한다.
        "response": tokenizer.decode(response_ids, skip_special_tokens=True).strip(),
        "generated_tokens": len(response_ids),
    }


### 학습 전 응답 기록

변화를 비교할 기준으로 기본 모델의 응답을 기록한다.
검증 질문 다섯 건의 질문·참고 응답·모델 응답을 모으며, 학습 후에도 같은 질문을 사용한다.


In [ ]:
def collect_responses(model, rows):
    return pd.DataFrame([
        {
            "source_row": row["source_row"],
            "question": row["instruction"],
            "reference": row["output"],
            **generate_response(model, row["instruction"]),
        }
        for row in rows
    ])


before_result = collect_responses(base, comparison_rows)
with pd.option_context("display.max_colwidth", None):
    display(before_result)


## 학습 데이터 구성

지시문·질문과 데이터셋의 응답(`output`)을 모델에 전달할 토큰으로 변환한다.
지시문·질문은 답변을 위한 문맥으로 사용하고, 응답을 예측한 오차만 학습에 반영한다.


In [ ]:
class ResponseDataset(Dataset):
    def __init__(self, rows):
        self.items = []
        for row in rows:
            messages = messages_for(row["instruction"])

            # 지시문·질문과 답변 시작 표시
            prompt_token_ids = tokenizer.apply_chat_template(
                messages,
                return_dict=False,
                add_generation_prompt=True,
            )

            # 정답 응답과 종료 표시까지 포함한 전체 입력
            full_token_ids = tokenizer.apply_chat_template(
                messages + [{"role": "assistant", "content": row["output"]}],
                return_dict=False,
            )

            # 응답 앞부분이 생성에 사용하는 입력과 같은지 확인한다.
            assert full_token_ids[:len(prompt_token_ids)] == prompt_token_ids

            # -100은 손실 계산에서 제외할 위치다. 응답 부분에는 실제 토큰 ID를 둔다.
            # 모델 내부에서 각 위치의 예측을 다음 위치의 정답과 비교한다.
            labels = (
                [-100] * len(prompt_token_ids)
                + full_token_ids[len(prompt_token_ids):]
            )
            self.items.append({
                "input_ids": full_token_ids,
                "attention_mask": [1] * len(full_token_ids),
                "labels": labels,
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):

        # 한 사례: input_ids, attention_mask, labels가 각각 길이 T인 정수 목록이다.
        return self.items[index]


train_dataset, val_dataset = ResponseDataset(train_rows), ResponseDataset(val_rows)


In [ ]:
# 배치로 묶기 위해 가장 긴 사례에 맞춰 패딩한다.
def collate_responses(items):
    length = max(len(item["input_ids"]) for item in items)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for item in items:
        pad = length - len(item["input_ids"])
        batch["input_ids"].append(item["input_ids"] + [tokenizer.pad_token_id] * pad)
        batch["attention_mask"].append(item["attention_mask"] + [0] * pad)
        batch["labels"].append(item["labels"] + [-100] * pad)

    # 각 값은 torch.long 텐서이며 모양은 (배치 크기, 배치 내 최대 토큰 길이)다.
    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}


## LoRA 어댑터 학습

### 모델 구성과 LoRA 적용 대상

각 Transformer 블록에서 `q_proj`·`k_proj`·`v_proj`는 Query·Key·Value를 만들고, `o_proj`는 Attention 결과를 출력 표현으로 변환한다.
FFN의 `gate_proj`·`up_proj`·`down_proj`는 각 토큰의 표현을 변환한다.
정규화 층은 표현의 크기를 조절하며, 블록 밖에는 토큰을 벡터로 바꾸는 임베딩과 다음 토큰의 점수를 계산하는 LM Head가 있다.

여기서는 모든 블록의 `q_proj`와 `v_proj`에 LoRA를 적용한다.
선택한 각 선형층에 독립적인 A·B 한 쌍을 추가하고, 기존 가중치와 편향은 모두 고정한다.


In [ ]:
TARGET_MODULES = ["q_proj", "v_proj"]

# 모델 전체 구조에서 LoRA를 적용할 층의 위치를 살펴본다.
print(base)


### LoRA 설정

`TARGET_MODULES`에 지정한 층에 어댑터를 연결한다.
rank는 A·B의 중간 크기이고, `alpha / rank`는 추가 변화량에 곱하는 배율이다.


In [ ]:
# LoRA의 중간 차원 수
LORA_RANK = 4

# 추가 변화량에 곱하는 alpha/r를 2로 유지한다.
LORA_ALPHA = 2 * LORA_RANK
LEARNING_RATE = 2e-4
EPOCHS = 3
output_dir = OUTPUT_ROOT / "outputs" / "koalpaca_lora"

# 프롬프트 평가 뒤 어댑터 초기화의 난수 시드를 고정한다.
set_seed(SEED)
base.config.use_cache = False
config = LoraConfig(

    # 다음 토큰 예측을 위한 언어 모델
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,

    # 어댑터 입력의 Dropout 확률
    lora_dropout=0.05,

    # LoRA를 적용할 층 이름
    target_modules=TARGET_MODULES,
)
model = get_peft_model(base, config)

# 전체 파라미터 수와 학습할 파라미터 수·비율을 출력한다.
model.print_trainable_parameters()


### 학습 설정

학습량과 메모리 사용량을 고려해 학습 설정을 지정한다.


In [ ]:
args = TrainingArguments(
    output_dir=str(output_dir / "checkpoints"),

    # GPU 한 개에서 한 번에 처리할 학습 사례 수
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # 메모리 사용을 줄이도록 작은 배치 8번의 기울기를 모아 한 번 갱신한다.
    gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,

    # 처음부터 같은 학습률을 사용한다.
    lr_scheduler_type="constant",

    # 중간 계산값 일부를 역전파 때 다시 계산해 메모리를 줄인다.
    activation_checkpointing=True,
    bf16=True,

    # 에포크마다 검증 손실을 계산한다.
    eval_strategy="epoch",

    # 마지막 어댑터를 별도 셀에서 저장한다.
    save_strategy="no",

    # 갱신 5번마다 손실 기록
    logging_steps=5,
    seed=SEED,
)


### 학습 실행

`trainer.train()`으로 정답 응답의 예측 오차를 줄이도록 LoRA 어댑터를 학습한다.
같은 검증 자료로 학습 전후 손실을 측정한다.


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    # 사례 목록을 하나의 텐서 배치로 만드는 함수
    data_collator=collate_responses,
    processing_class=tokenizer,
)
before_loss = trainer.evaluate()["eval_loss"]
trainer.train()
after_loss = trainer.evaluate()["eval_loss"]


## 파인튜닝 결과 비교

### 같은 질문의 학습 전후 응답

파인튜닝에 따른 답변의 변화를 확인하도록 같은 검증 질문과 생성 규칙으로 학습 전후 응답을 나란히 표시한다.


In [ ]:
model.config.use_cache = True
model.activation_checkpointing_disable()
after_result = collect_responses(model, comparison_rows)
comparison = before_result[
    ["source_row", "question", "reference", "response"]
].rename(columns={"response": "before"})
comparison = comparison.merge(
    after_result[["source_row", "response"]].rename(columns={"response": "after"}),
    on="source_row",
)
with pd.option_context("display.max_colwidth", None):
    display(comparison)
display(
    pd.concat([
        before_result.assign(stage="before"),
        after_result.assign(stage="after"),
    ])[["stage", "source_row", "generated_tokens"]]
)


### 실제 손실 기록 읽기

학습 손실의 추이와 학습 전후 검증 손실을 그려 정답 응답을 예측하는 정도의 변화를 확인한다.
손실이 낮을수록 정답 토큰에 부여한 확률이 전반적으로 높다.


In [ ]:
loss_log = pd.DataFrame([
    {"step": row["step"], "loss": row["loss"]}
    for row in trainer.state.log_history
    if "loss" in row
])
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
axes[0].plot(loss_log["step"], loss_log["loss"], marker="o")
axes[0].set(xlabel="Optimizer step", ylabel="Training loss", title="Training log")
axes[1].bar(["Before", "After"], [before_loss, after_loss])
axes[1].set(ylabel="Validation loss", title="Same validation examples")
plt.show()
print({"validation_loss_before": before_loss, "validation_loss_after": after_loss})


## 어댑터 저장과 복원

### 학습 결과 저장

다시 학습하지 않고 사용할 수 있도록 어댑터와 토크나이저를 저장한다.
어댑터에는 학습한 변화량과 적용 설정이 담기며, 사용할 때 같은 기본 모델에 연결한다.


In [ ]:
adapter_dir = output_dir / "adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)


In [ ]:
# 기본 모델과 입력·생성 설정을 복원하기 위해 저장한다.
metadata = {
    "base_model": MODEL_ID,
    "dtype": str(DTYPE).removeprefix("torch."),
    "system": SYSTEM,
    "generation_config": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "eos_token_id": model.generation_config.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    },
}
(adapter_dir / "run_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)


### 저장한 어댑터 복원

저장한 어댑터를 다시 사용하도록 기본 모델에 연결하고, 토크나이저와 입력·생성 설정을 복원한다.
GPU 메모리를 확보하기 위해 기존 모델을 먼저 해제한다.


In [ ]:
def load_saved_adapter(saved_adapter_dir):
    info = json.loads(
        (saved_adapter_dir / "run_metadata.json").read_text(encoding="utf-8")
    )
    saved_tokenizer = AutoTokenizer.from_pretrained(saved_adapter_dir)
    saved_base = AutoModelForCausalLM.from_pretrained(
        info["base_model"],

        # 저장된 자료형 이름 "bfloat16"을 torch.bfloat16으로 바꾼다.
        dtype=getattr(torch, info["dtype"]),
        device_map={"": 0},
    )
    saved_base.config.pad_token_id = saved_tokenizer.pad_token_id
    saved_base.generation_config.pad_token_id = saved_tokenizer.pad_token_id

    # 추론용으로 복원한다.
    loaded_model = PeftModel.from_pretrained(saved_base, saved_adapter_dir)
    loaded_model.config.use_cache = True
    loaded_model.generation_config.eos_token_id = (
        info["generation_config"]["eos_token_id"]
    )
    assert saved_tokenizer.pad_token_id == info["generation_config"]["pad_token_id"]
    return loaded_model, saved_tokenizer, info


del trainer, model, base
gc.collect()
torch.cuda.empty_cache()

reloaded_model, tokenizer, saved_metadata = load_saved_adapter(adapter_dir)
SYSTEM = saved_metadata["system"]
MAX_NEW_TOKENS = saved_metadata["generation_config"]["max_new_tokens"]


### 새 질문에 응답 생성

복원한 모델에 비교에 쓰지 않은 새 질문을 넣어, 요구한 내용과 형식을 따르는지 확인한다.


In [ ]:
QUESTION = "도서관에서 빌린 책을 잘 관리하는 방법을 세 가지 알려주세요."
print("질문:", QUESTION)
print("응답:", generate_response(reloaded_model, QUESTION))
